# Agent and tool capstone

This notebook uses **frozen planner trajectories, not live model output**, for a deterministic HR/IT onboarding agent. Every side effect is validated, authorized, approved, budgeted, recorded, and verified.

In [ ]:
import importlib.util
import json
from copy import deepcopy
from pathlib import Path

ROOT = Path.cwd()
spec = importlib.util.spec_from_file_location("capstone_lab", ROOT / "capstone_lab.py")
capstone = importlib.util.module_from_spec(spec)
import sys
sys.modules[spec.name] = capstone
spec.loader.exec_module(capstone)

def load(name):
    return json.loads((ROOT / "fixtures" / name).read_text())

sessions = [capstone.Session.from_dict(item) for item in load("sessions.json")]
capabilities = {item["tool"]: capstone.Capability.from_dict(item) for item in load("capabilities.json")}
trajectories = [capstone.Trajectory.from_dict(item) for item in load("trajectories.json")]
tool_outputs = load("tool_outputs.json")
expected = load("expected_terminals.json")
approval_records = load("approvals.json")
envelope = load("envelope.json")
budgets = load("budget.json")
today = capstone.date.fromisoformat(load("clock.json")["today"])

def make_state(with_approvals=True):
    record = load("state.json")
    if with_approvals:
        record["approvals"] = {item["approval_id"]: item for item in approval_records if item["approval_id"] in {
            "approve-create-1", "approve-group-1", "approve-order-1", "approve-email-1", "approve-reconcile"
        }}
    return capstone.AuthorizationState.from_dict(record)

def run(item, current=None):
    session = sessions[3] if item.trajectory_id == "T7" else sessions[0]
    limit = budgets["T3"] if item.trajectory_id == "T3" else budgets["default"]
    return capstone.run_trajectory(item, session, current or make_state(), capabilities, capstone.Budget(**limit), tool_outputs, today)

print("scenario: new-hire onboarding", "tenants:", [session.tenant for session in sessions[:2]])

## 2. Dynamic capability exposure by phase and role

The trusted role and current phase determine which tools can be proposed for execution. This map is not inferred from planner text.

In [ ]:
employee = sessions[2]
print("employee lookup:", sorted(capstone.exposed_capabilities(employee, make_state(), "lookup", capabilities)))
print("HR lookup:", sorted(capstone.exposed_capabilities(sessions[0], make_state(), "lookup", capabilities)))
print("HR provision:", sorted(capstone.exposed_capabilities(sessions[0], make_state(), "provision", capabilities)))
print("HR notify:", sorted(capstone.exposed_capabilities(sessions[0], make_state(), "notify", capabilities)))
print("notify before provision side effect:", "send_welcome_email" in capstone.exposed_capabilities(sessions[0], make_state(), "provision", capabilities))

## 3. Happy path with approvals, receipts, and audit

The trajectory is frozen fixture data. A pre-existing account demonstrates idempotent replay while later approved side effects write receipts and correlation-linked audit events.

In [ ]:
happy = run(trajectories[0])
print("terminal:", happy.terminal.value)
print("decisions:", [(item.step_id, item.decision.value, item.reason_codes) for item in happy.decisions])
print("audit events:", len(happy.audit), "receipts:", len(make_state().receipts))

## 4. Approval required, then re-evaluate

Missing approval is an escalation, not an assumption. The pending fingerprint is approved and the complete trajectory is re-run.

In [ ]:
approval_state = make_state()
pending = run(trajectories[1], approval_state)
print("before approval:", pending.terminal.value, pending.decisions[-1].pending_fingerprint)
approval_state.approvals["approve-t2"] = capstone.Approval(
    "approve-t2", pending.decisions[-1].pending_fingerprint, "hr_bri", "hr_admin", "bu-north", capstone.date(2027, 1, 1)
)
recovered = run(trajectories[1], approval_state)
print("after approval and re-evaluation:", recovered.terminal.value)

## 5. Budgets and loop detection

Turn, spend, and side-effect budgets are independent limits. A repeated step fingerprint reaches the loop terminal on its third proposal.

In [ ]:
over_budget = run(trajectories[2])
loop = run(trajectories[3])
print("budget:", over_budget.terminal.value, over_budget.decisions[-1].reason_codes)
print("loop:", loop.terminal.value, loop.decisions[-1].reason_codes)

## 6. Tool-output injection cannot add steps

The read result contains an instruction to add an employee to `admins`, but its provenance is `tool_output`; the gateway blocks it and continues the original trajectory.

In [ ]:
injection = run(trajectories[4])
print("terminal:", injection.terminal.value)
print("blocked tool-output step:", injection.decisions[-1].reason_codes, "blocked attempts:", injection.blocked_attempts)

## 7. Execution-time revocation and cross-tenant binding

Authorization is checked immediately before execution. A revoked session and a tenant mismatch both stop before any side effect.

In [ ]:
revoked = capstone.run_trajectory(trajectories[6], sessions[3], make_state(), capabilities, capstone.Budget(**budgets["default"]), tool_outputs, today)
cross_tenant = run(trajectories[5])
print("revoked:", revoked.terminal.value, revoked.decisions[-1].reason_codes)
print("cross tenant:", cross_tenant.terminal.value, cross_tenant.decisions[-1].reason_codes)

## 8. Post-action verification and reconciliation

A receipt does not replace a read-back. The fixture simulates a system-of-record disagreement and requires reconciliation.

In [ ]:
mismatch = run(trajectories[7])
print("terminal:", mismatch.terminal.value)
print("verification decision:", mismatch.decisions[-1].reason_codes)
print("audit complete:", mismatch.audit_complete)

## 9. Trajectory evaluation and capstone gate

The gate combines terminal accuracy, authorization, approval, audit, and spend evidence. Removing approvals makes the same frozen trajectories fail rather than silently assuming authorization.

In [ ]:
baseline_results = [run(item) for item in trajectories]
evaluation_state = make_state()
evaluation = capstone.evaluate(trajectories, baseline_results, expected, evaluation_state, capabilities)
gate = capstone.capstone_gate(evaluation, envelope)
print("terminal accuracy:", evaluation["terminal_accuracy"])
print("gate:", gate)
print("gate passes:", capstone.gate_passes(gate))
without_approvals = make_state(with_approvals=False)
broken_results = [run(item, without_approvals) for item in trajectories]
broken = capstone.evaluate(trajectories, broken_results, expected, without_approvals, capabilities)
print("without approvals:", broken["terminal_accuracy"], "approval violations:", broken["approval_violations"], "gate passes:", capstone.gate_passes(capstone.capstone_gate(broken, envelope)))
evaluation_state.approvals.clear()
tampered = capstone.evaluate(trajectories, baseline_results, expected, evaluation_state, capabilities)
print("checker after approval tampering:", tampered["approval_violations"], "gate passes:", capstone.gate_passes(capstone.capstone_gate(tampered, envelope)))
print("the checker reads evidence, it doesn't trust the runner.")

## Exercises

1. Add an approval with the wrong fingerprint and observe that it cannot authorize the step.
2. Reduce the side-effect budget and explain which terminal state changes.
3. Add a reconciliation mismatch to a group or order and preserve the audit evidence.